# Task 08 – Graph Intelligence Dashboard

Create the required Streamlit dashboard and verify its input files.

In [1]:
!pip install -q torch-geometric ogb streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 38.9 MB/s eta 0:00:00


In [2]:
import os
# Compatibility fix for trusted PyG objects downloaded by the official OGB package on PyTorch 2.6+.
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/OGBN_Arxiv_Project')
RESULTS_ROOT = PROJECT_ROOT / 'results'
MODELS_DIR = PROJECT_ROOT / 'models'
SRC_DIR = PROJECT_ROOT / 'src'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'

for folder in [PROJECT_ROOT, RESULTS_ROOT, MODELS_DIR, SRC_DIR, ARTIFACTS_DIR,
               PROJECT_ROOT / 'notebooks', PROJECT_ROOT / 'visualizations',
               PROJECT_ROOT / 'dashboard', PROJECT_ROOT / 'report',
               PROJECT_ROOT / 'presentation', PROJECT_ROOT / 'video']:
    folder.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)

Mounted at /content/drive
Project root: /content/drive/MyDrive/OGBN_Arxiv_Project


In [4]:
# Common reproducibility settings
import warnings
import random
import numpy as np

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [5]:
# Libraries required for Task 08
import streamlit as st
import pandas as pd
from pathlib import Path

OUTPUT = RESULTS_ROOT / 'task08_dashboard'
OUTPUT.mkdir(parents=True, exist_ok=True)

## 8.1 Dashboard Purpose and Information Sources

The Streamlit dashboard presents graph statistics, model performance, node classifications, and embedding visualizations in one interface. It uses the saved outputs from graph analysis, model training, evaluation, and explainability so the displayed evidence remains consistent with the notebook results.

## 8.2 Streamlit Dashboard Application

Creates a multi-page dashboard that presents graph statistics, model performance, node-level classifications, and embedding visualizations.

In [6]:
dashboard_code = '\nimport json\nfrom pathlib import Path\nimport pandas as pd\nimport streamlit as st\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nTASK02 = PROJECT_ROOT / \'results\' / \'task02_graph_analysis\'\nTASK05 = PROJECT_ROOT / \'results\' / \'task05_training\'\nTASK06 = PROJECT_ROOT / \'results\' / \'task06_evaluation\'\nTASK07 = PROJECT_ROOT / \'results\' / \'task07_explainability\'\n\nst.set_page_config(page_title=\'OGBN-Arxiv Dashboard\', layout=\'wide\')\nst.title(\'OGBN-Arxiv Graph Intelligence Dashboard\')\npage = st.sidebar.radio(\'Page\', [\'Graph statistics\', \'Model performance\', \'Node classification\', \'Embeddings\'])\n\nif page == \'Graph statistics\':\n    st.header(\'Graph statistics\')\n    stats = json.loads((TASK02 / \'graph_statistics.json\').read_text())\n    a,b,c,d = st.columns(4)\n    a.metric(\'Papers\', f"{stats[\'nodes\']:,}"); b.metric(\'Citations\', f"{stats[\'edges\']:,}")\n    c.metric(\'Features\', stats[\'features\']); d.metric(\'Classes\', stats[\'classes\'])\n    st.metric(\'Graph density\', f"{stats[\'density\']:.10f}")\n    st.image(str(TASK02 / \'degree_distribution.png\'))\n    st.image(str(TASK02 / \'sample_subgraph.png\'))\nelif page == \'Model performance\':\n    st.header(\'Validation and test metrics\')\n    results = pd.read_csv(TASK06 / \'model_evaluation.csv\')\n    st.dataframe(results, use_container_width=True)\n    st.bar_chart(results[results.split == \'Test\'].set_index(\'model\')[[\'accuracy\',\'precision_macro\',\'recall_macro\',\'f1_macro\']])\n    st.image(str(TASK05 / \'best_model_accuracy_curves.png\'))\nelif page == \'Node classification\':\n    results = pd.read_csv(TASK06 / \'node_classification_results.csv\')\n    model = st.selectbox(\'Model\', sorted(results.model.unique()))\n    selected = results[results.model == model]\n    node = st.selectbox(\'Node ID\', selected.node_id.tolist())\n    st.dataframe(selected[selected.node_id == node], use_container_width=True)\nelse:\n    st.header(\'PCA node embeddings\')\n    st.image(str(TASK07 / \'pca_embeddings.png\'))\n    st.dataframe(pd.read_csv(TASK07 / \'pca_embeddings.csv\').head(100), use_container_width=True)\n'
app_path = PROJECT_ROOT / 'dashboard' / 'app.py'
app_path.write_text(dashboard_code)
print('Dashboard created:', app_path)

Dashboard created: /content/drive/MyDrive/OGBN_Arxiv_Project/dashboard/app.py


## 8.3 Dashboard Output

The dashboard presents the required graph statistics, model metrics, node-classification results, and PCA embedding visualization. Its values come from the files produced by Tasks 02, 05, 06, and 07.